# Global snowmelt lapse rates

Onset delay per 100 m of elevation from the continental latitude–elevation bins: multiple
regression on latitude + elevation, count-weighted per-latitude lapse rates, and the
latitude-rate variant. Same cube and thresholds as `lat_elev_binning.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from gsro_analysis import aggregate, paths, settings

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

PIX_THRESH = 1000   # a (continent, latitude, elevation) bin needs > 1000 pixels to be shown

# the continents cube: continent x latitude x elevation x chili_class x water_year, with
# <var> = bin mean, <var>_n = pixel count (see gsro_analysis.aggregate). No 'statistic' axis.
continents_ds = aggregate.open_aggregate('continents', config.version)

# all insolation classes together (count-weighted), thresholded
ds = aggregate.threshold(aggregate.collapse(continents_ds), PIX_THRESH)

# sunny (warm) minus shaded (cool) CHILI classes, each thresholded on its own count
cool = aggregate.threshold(continents_ds.sel(chili_class='cool'), PIX_THRESH)
warm = aggregate.threshold(continents_ds.sel(chili_class='warm'), PIX_THRESH)
ds['chili_warm_cool_difference'] = warm['runoff_onset_median'] - cool['runoff_onset_median']
ds['chili_warm_cool_ratio'] = warm['runoff_onset_median'] / cool['runoff_onset_median']
ds['chili_warm_cool_n'] = xr.ufuncs.minimum(warm['runoff_onset_median_n'], cool['runoff_onset_median_n'])

# GTOPO30 land-pixel histogram: the grey background of every panel (all land, not just mapped pixels)
dem_pixel_count = continents_ds['dem_pixel_count']
ds

In [ ]:
continent_order = ['North America', 'Europe', 'Asia', 'South America', 'Africa', 'Oceania']
ds = ds.reindex({'continent': continent_order})
ds

## Multiple regression: onset vs |latitude| + elevation

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

runoff_data = ds.runoff_onset_median

# Convert to DataFrame for easier analysis
data_points = []
for cont in runoff_data.continent.values:
    for lat in runoff_data.latitude.values:
        for elev in runoff_data.elevation.values:
            value = runoff_data.sel(continent=cont, latitude=lat, elevation=elev).values
            # Check if the value is valid (not NaN)
            if not np.isnan(value):
                data_points.append({
                    'continent': cont,
                    'latitude': lat,
                    'elevation': elev,
                    'runoff_onset': float(value)
                })

# Create DataFrame
df = pd.DataFrame(data_points)
print(f"Total data points: {len(df)}")
print(df.head())

# Analyze by hemisphere (important because seasonal patterns are reversed)
df['hemisphere'] = np.where(df['latitude'] >= 0, 'Northern', 'Southern')
df['abs_latitude'] = np.abs(df['latitude'])

print("\nData points by hemisphere:")
print(df['hemisphere'].value_counts())

# Function to perform regression analysis
def analyze_relationship(data, x_vars, y_var):
    """
    Perform multiple linear regression and return results
    """
    X = data[x_vars]
    y = data[y_var]
    
    # Add constant for statsmodels
    X_with_const = sm.add_constant(X)
    
    # Fit model
    model = sm.OLS(y, X_with_const).fit()
    
    # Calculate R-squared using sklearn for consistency
    y_pred = model.predict(X_with_const)
    r2 = r2_score(y, y_pred)
    
    return model, r2

# Analysis for each hemisphere separately
hemispheres = ['Northern', 'Southern', 'All']
results = {}

for hemi in hemispheres:
    if hemi == 'All':
        data = df
    else:
        data = df[df['hemisphere'] == hemi]
    
    # Perform regression
    model, r2 = analyze_relationship(
        data, 
        ['abs_latitude', 'elevation'], 
        'runoff_onset'
    )
    
    # Store results
    results[hemi] = {
        'model': model,
        'r2': r2,
        'data_points': len(data)
    }
    
    # Print results
    print(f"\n--- {hemi} Hemisphere Results ({len(data)} data points) ---")
    print(model.summary().tables[1])
    print(f"R-squared: {r2:.3f}")
    
    # Extract coefficients for easy interpretation
    lat_coef = model.params['abs_latitude']
    elev_coef = model.params['elevation']
    
    print(f"\nInterpretation:")
    print(f"- For every 1° increase in latitude (away from equator), "
          f"runoff onset changes by {lat_coef:.2f} days")
    print(f"- For every 100m increase in elevation, "
          f"runoff onset changes by {elev_coef * 100:.2f} days")

# Analyze by continent
print("\n\n--- Analysis by Continent ---")
continent_results = {}

for cont in df['continent'].unique():
    cont_data = df[df['continent'] == cont]
    model, r2 = analyze_relationship(
        cont_data, 
        ['abs_latitude', 'elevation'], 
        'runoff_onset'
    )
    
    continent_results[cont] = {
        'model': model,
        'r2': r2,
        'lat_coef': model.params['abs_latitude'],
        'elev_coef': model.params['elevation'],
        'data_points': len(cont_data)
    }
    
    print(f"\n--- {cont} Results ({len(cont_data)} data points) ---")
    print(f"Latitude effect (days per degree): {model.params['abs_latitude']:.2f}")
    print(f"Elevation effect (days per 100m): {model.params['elevation'] * 100:.2f}")
    print(f"R-squared: {r2:.3f}")

In [ ]:
# Extract the data we need for analysis - mean values and count values
runoff_data = ds.runoff_onset_median
count_data = ds.runoff_onset_median_n.where(ds.runoff_onset_median.notnull())

# Convert to DataFrame for easier analysis with weights
data_points = []
for cont in runoff_data.continent.values:
    for lat in runoff_data.latitude.values:
        for elev in runoff_data.elevation.values:
            value = runoff_data.sel(continent=cont, latitude=lat, elevation=elev).values
            count = count_data.sel(continent=cont, latitude=lat, elevation=elev).values
            
            # Check if the value is valid (not NaN) and has a valid count
            if not np.isnan(value) and not np.isnan(count) and count > 0:
                data_points.append({
                    'continent': cont,
                    'latitude': lat,
                    'elevation': elev,
                    'runoff_onset': float(value),
                    'count': float(count)  # Add count as a weight
                })

# Create DataFrame
df = pd.DataFrame(data_points)
print(f"Total data points: {len(df)}")
print(df.head())

# Analyze by hemisphere (important because seasonal patterns are reversed)
df['hemisphere'] = np.where(df['latitude'] >= 0, 'Northern', 'Southern')
df['abs_latitude'] = np.abs(df['latitude'])

print("\nData points by hemisphere:")
print(df['hemisphere'].value_counts())

# Function to perform weighted regression analysis
def analyze_relationship_weighted(data, x_vars, y_var, weight_var):
    """
    Perform weighted multiple linear regression and return results
    """
    X = data[x_vars]
    y = data[y_var]
    weights = data[weight_var]
    
    # Normalize weights to sum to 1
    normalized_weights = weights / weights.sum()
    
    # Add constant for statsmodels
    X_with_const = sm.add_constant(X)
    
    # Fit model with weights
    model = sm.WLS(y, X_with_const, weights=normalized_weights).fit()
    
    # Calculate weighted R-squared
    y_pred = model.predict(X_with_const)
    weighted_residuals = (y - y_pred) * np.sqrt(normalized_weights)
    weighted_total = (y - np.average(y, weights=normalized_weights)) * np.sqrt(normalized_weights)
    r2 = 1 - (np.sum(weighted_residuals**2) / np.sum(weighted_total**2))
    
    return model, r2

# Analysis for each hemisphere separately
hemispheres = ['Northern', 'Southern', 'All']
results = {}

for hemi in hemispheres:
    if hemi == 'All':
        data = df
    else:
        data = df[df['hemisphere'] == hemi]
    
    # Perform weighted regression
    model, r2 = analyze_relationship_weighted(
        data, 
        ['abs_latitude', 'elevation'], 
        'runoff_onset',
        'count'  # Use count as weights
    )
    
    # Store results
    results[hemi] = {
        'model': model,
        'r2': r2,
        'data_points': len(data)
    }
    
    # Print results
    print(f"\n--- {hemi} Hemisphere Results ({len(data)} data points) ---")
    print(model.summary().tables[1])
    print(f"Weighted R-squared: {r2:.3f}")
    
    # Extract coefficients for easy interpretation
    lat_coef = model.params['abs_latitude']
    elev_coef = model.params['elevation']
    
    print(f"\nInterpretation:")
    print(f"- For every 1° increase in latitude (away from equator), "
          f"runoff onset changes by {lat_coef:.2f} days")
    print(f"- For every 100m increase in elevation, "
          f"runoff onset changes by {elev_coef * 100:.2f} days")

# Analyze by continent
print("\n\n--- Analysis by Continent ---")
continent_results = {}

for cont in df['continent'].unique():
    cont_data = df[df['continent'] == cont]
    
    # Skip if insufficient data
    if len(cont_data) < 3:
        print(f"\n--- {cont}: Insufficient data for analysis ---")
        continue
        
    model, r2 = analyze_relationship_weighted(
        cont_data, 
        ['abs_latitude', 'elevation'], 
        'runoff_onset',
        'count'  # Use count as weights
    )
    
    continent_results[cont] = {
        'model': model,
        'r2': r2,
        'lat_coef': model.params['abs_latitude'],
        'elev_coef': model.params['elevation'],
        'data_points': len(cont_data)
    }
    
    print(f"\n--- {cont} Results ({len(cont_data)} data points) ---")
    print(f"Latitude effect (days per degree): {model.params['abs_latitude']:.2f}")
    print(f"Elevation effect (days per 100m): {model.params['elevation'] * 100:.2f}")
    print(f"Weighted R-squared: {r2:.3f}")

## Count-weighted lapse rate per latitude band

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

def weighted_polyfit(da, dim, weights, deg=1):
    """
    Perform weighted linear regression along a dimension.
    
    Parameters:
    -----------
    da : xarray.DataArray
        The data to fit
    dim : str
        Dimension to fit along
    weights : xarray.DataArray
        Weights for the fit (e.g., count data)
    deg : int
        Degree of polynomial (default: 1 for linear)
        
    Returns:
    --------
    xarray.DataArray
        DataArray with coefficients
    """
    # Extract the dimension values as predictors
    x_values = da[dim].values
    
    # Move the fit dimension to the end for easier processing
    da_trans = da.transpose(..., dim)
    weights_trans = weights.transpose(..., dim)
    
    # Create output structure - will have a 'degree' dimension
    out_dims = list(da_trans.dims[:-1]) + ['degree']
    out_shape = list(da_trans.shape[:-1]) + [deg + 1]
    out_coords = {k: da_trans[k] for k in da_trans.dims[:-1]}
    out_coords['degree'] = np.arange(deg + 1)
    
    result = np.zeros(out_shape)
    
    # Iterate over all combinations of dimensions except the fit dimension
    if len(da_trans.dims) > 1:
        # Get indices for all dimensions except the last
        other_dims = da_trans.dims[:-1]
        indices = [range(da_trans.sizes[d]) for d in other_dims]
        
        # Iterate over all combinations
        for idx in np.ndindex(*[da_trans.sizes[d] for d in other_dims]):
            # Extract 1D arrays for this combination
            y = da_trans.values[idx]
            w = weights_trans.values[idx]
            
            # Handle missing values
            mask = ~np.isnan(y) & ~np.isnan(w) & (w > 0)
            if np.sum(mask) > deg:  # Need at least deg+1 points
                # Normalize weights to sum to 1
                w_masked = w[mask] / np.sum(w[mask])
                
                # For linear regression (deg=1)
                if deg == 1:
                    model = LinearRegression()
                    model.fit(x_values[mask].reshape(-1, 1), y[mask], sample_weight=w_masked)
                    result[idx][0] = model.intercept_
                    result[idx][1] = model.coef_[0]
                else:
                    # For higher degree polynomials
                    coeffs = np.polyfit(x_values[mask], y[mask], deg, w=w_masked)
                    result[idx] = coeffs[::-1]  # Reverse to match polyfit convention
            else:
                result[idx] = np.nan
    else:
        # Handle the 1D case
        y = da_trans.values
        w = weights_trans.values
        
        # Handle missing values
        mask = ~np.isnan(y) & ~np.isnan(w) & (w > 0)
        if np.sum(mask) > deg:  # Need at least deg+1 points
            # Normalize weights to sum to 1
            w_masked = w[mask] / np.sum(w[mask])
            
            # For linear regression (deg=1)
            if deg == 1:
                model = LinearRegression()
                model.fit(x_values[mask].reshape(-1, 1), y[mask], sample_weight=w_masked)
                result[0] = model.intercept_
                result[1] = model.coef_[0]
            else:
                # For higher degree polynomials
                coeffs = np.polyfit(x_values[mask], y[mask], deg, w=w_masked)
                result = coeffs[::-1]  # Reverse to match polyfit convention
        else:
            result[:] = np.nan
    
    # Create the output DataArray
    output = xr.DataArray(result, dims=out_dims, coords=out_coords)
    return output

In [ ]:
# Use the function instead of polyfit
snowmelt_runoff_onset_lapse_rates_da = weighted_polyfit(
    ds['runoff_onset_median'], 
    'elevation', 
    ds['runoff_onset_median_n'].where(ds['runoff_onset_median'].notnull()), 
    deg=1
).sel(degree=1).squeeze()*100
snowmelt_runoff_onset_lapse_rates_da

In [ ]:
f,ax=plt.subplots(figsize=(5,10))
snowmelt_runoff_onset_lapse_rates_da.plot.line(ax=ax, y='latitude')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)

In [ ]:
polyfit_da = ds['runoff_onset_median'].where(lambda x: x.count(dim='elevation')>10).polyfit(dim='elevation', deg=1)['polyfit_coefficients']
polyfit_da

In [ ]:
snowmelt_runoff_onset_lapse_rates_da = polyfit_da.sel(degree=1).squeeze()*100
snowmelt_runoff_onset_lapse_rates_da

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create FacetGrid
df = ds['runoff_onset_median'].to_dataframe().reset_index()
g = sns.FacetGrid(df, col='continent', col_wrap=3, height=4, aspect=1.5)

# Plot lines
g.map_dataframe(sns.lineplot, x='elevation', y='runoff_onset_median', 
                hue='latitude', palette='viridis')

# Create colorbar
norm = plt.Normalize(df.latitude.min(), df.latitude.max())
sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])

# Add colorbar to the figure
cax = g.fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_label('Latitude (°)', rotation=270, labelpad=15)

# Adjust layout
g.fig.subplots_adjust(right=0.9)
plt.show()

f.savefig(paths.figdir('global', config.version) / 'lapse_rates_by_latitude_groupby_continents.png', dpi=300)

def calculate_weighted_lapse_rates(ds):
    lapse_rates = xr.Dataset()
    r_squared = xr.Dataset()
    
    for continent in ds.continent.values:
        ds_cont = ds.sel(continent=continent)
        count_threshold = 100000
        
        # Get arrays for calculations
        counts = ds_cont['runoff_onset_median_n'].astype(float)
        valid_points = counts > count_threshold
        onset_means = ds_cont['runoff_onset_median'].where(valid_points)
        
        lapse_rates_cont = []
        r2_values_cont = []
        
        for lat in ds_cont.latitude.values:
            elev = ds_cont.elevation.values
            onset = onset_means.sel(latitude=lat)
            weights = np.sqrt(counts.sel(latitude=lat))  # Square root of counts as weights
            
            mask = ~np.isnan(onset) & ~np.isnan(weights)
            if mask.sum() >= 2:
                # Weighted polyfit
                coeffs = np.polyfit(elev[mask], onset[mask], 1, w=weights[mask])
                slope = coeffs[0]
                
                # Calculate R-squared
                y_pred = np.polyval(coeffs, elev[mask])
                ss_tot = np.sum(weights[mask] * (onset[mask] - np.average(onset[mask], weights=weights[mask]))**2)
                ss_res = np.sum(weights[mask] * (onset[mask] - y_pred)**2)
                r2 = 1 - (ss_res / ss_tot)
                
                lapse_rate = slope * 100  # Convert to days per 100m
            else:
                lapse_rate = np.nan
                r2 = np.nan
                
            lapse_rates_cont.append(lapse_rate)
            r2_values_cont.append(r2)
            
        lapse_rates[continent] = xr.DataArray(
            lapse_rates_cont,
            dims=['latitude'],
            coords={'latitude': ds_cont.latitude}
        )
        
        r_squared[continent] = xr.DataArray(
            r2_values_cont,
            dims=['latitude'],
            coords={'latitude': ds_cont.latitude}
        )
    
    return lapse_rates, r_squared


lapse_rates, r_squared = calculate_weighted_lapse_rates(ds)

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

# Define colormap
colors = plt.cm.Set2(np.linspace(0, 1, len(lapse_rates.data_vars)))

# Plot lapse rates
for i, continent in enumerate(lapse_rates.data_vars):
    mask = ~np.isnan(lapse_rates[continent]) & (r_squared[continent] > 0.3)
    ax1.plot(lapse_rates[continent][mask], 
             lapse_rates.latitude[mask], 
             label=continent, 
             color=colors[i],
             marker='o',
             markersize=4)

# Plot R-squared values
for i, continent in enumerate(r_squared.data_vars):
    mask = ~np.isnan(r_squared[continent])
    ax2.plot(r_squared[continent][mask], 
             r_squared.latitude[mask], 
             color=colors[i],
             marker='o',
             markersize=4)

# Customize left plot
ax1.axvline(x=0, color='k', linestyle='--', alpha=0.3)
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Days delay per 100m elevation gain')
ax1.set_ylabel('Latitude')
ax1.set_title('Runoff Onset Lapse Rates')
ax1.legend(bbox_to_anchor=(0, 1.02, 1, 0.2), 
          loc="lower left",
          mode="expand", 
          ncol=3)

# Customize right plot
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('R²')
ax2.set_title('Regression Quality')

plt.tight_layout()
plt.show()

f.savefig(paths.figdir('global', config.version) / 'global_lapse_rates_by_continent.png',dpi=300)

In [ ]:
snowmelt_runoff_onset_lapse_rates_df = snowmelt_runoff_onset_lapse_rates_da.to_dataframe().reset_index()
snowmelt_runoff_onset_lapse_rates_df

In [ ]:
f,ax=plt.subplots(figsize=(4,8),dpi=300,layout='constrained')

#snowmelt_runoff_onset_lapse_rates_da.plot.scatter(ax=ax,y='latitude')
levels, categories = pd.factorize(snowmelt_runoff_onset_lapse_rates_df['continent'])
colors = [plt.cm.tab10(i) for i in levels] # using the "tab10" colormap
handles = [matplotlib.patches.Patch(color=plt.cm.tab10(i), label=c) for i, c in enumerate(categories)]

snowmelt_runoff_onset_lapse_rates_df.plot.scatter(ax=ax,x='polyfit_coefficients',y='latitude',c=colors)

ax.axvline(x=0, color='black', linestyle='--', linewidth=1)

ax.legend(handles=handles, title='Continent', loc='center right')

lat_ticks = [-60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70, 80]
lat_labels = ['60°S', '50°S', '40°S', '30°S', '20°S', '10°S', '0°', '10°N', '20°N', '30°N', '40°N', '50°N', '60°N', '70°N', '80°N']

ax.set_yticks(lat_ticks)
ax.set_yticklabels(lat_labels)
ax.set_ylabel('Latitude [degrees]')
ax.set_xlabel('Delay per 100 m increase in elevation [days]')

ax.grid(True, which='major', linestyle='--', alpha=0.5)

## Latitude rates (days per degree, per elevation bin)

In [ ]:
# Use the function for latitude analysis
polyfit_da_lat = weighted_polyfit(
    ds['runoff_onset_median'], 
    'latitude', 
    ds['runoff_onset_median_n'].where(ds['runoff_onset_median'].notnull()), 
    deg=1
)

# Get days per degree latitude from the slope coefficient
snowmelt_runoff_onset_lat_rates_da = polyfit_da_lat.sel(degree=1)
# multiply by negative 1 for africa, oceania, and south america
snowmelt_runoff_onset_lat_rates_da.loc[dict(continent=['Africa', 'Oceania', 'South America'])] *= -1
snowmelt_runoff_onset_lat_rates_da

In [ ]:
f,ax=plt.subplots(figsize=(20,10))
snowmelt_runoff_onset_lat_rates_da.plot.line(ax=ax, y='elevation')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)